# Exercise: JSON Data Contracts

## Overview

A **data contract** is a formal agreement that defines the structure, rules, and expectations for a dataset. In this exercise, you will write a JSON data contract for a provided student grades dataset, then validate the dataset against your contract using Python.

---

## The Dataset

You have been provided with `student_grades.csv`. Open the file and examine its contents before writing your contract.

The dataset has four required columns:

| Column | Description | Type / Rule |
|---|---|---|
| **Student Name** | Full name of the student | `string` |
| **Class** | Course the student is enrolled in | `string`, restricted values |
| **Semester** | Semester in which the class was taken | `string`, restricted values |
| **Grade** | Student's numeric grade | `integer`, 60–100 inclusive |

### Approved values for `Class`

- Calculus 1
- English 101
- Python 101
- Intro to Databases

### Approved values for `Semester`

- Fall
- Spring

### Approved range for `Grade`

- 60 to 100, inclusive

---

## Part 1 — Write the JSON Data Contract

Create a file named:

```text
student_grades_contract.json
```

Your contract must include the following four sections:

### 1a. Contract Metadata

Include a top-level `contract` section with:

- `name`
- `version`
- `description`

### 1b. Source Definition

Include a `source` section with:

- `format`
- `delimiter`
- `has_header`

### 1c. Schema Definition

Include a `schema` section with a `fields` object.

Each column must define:

- `type`
- `nullable`

Additional rules:

- `Class` and `Semester` must include `allowed_values`
- `Grade` must include `min_value` and `max_value`

### 1d. Quality Rules

Include a `quality_rules` section with:

- `no_duplicate_rows`
- `min_row_count`

---

## Contract Skeleton

```json
{
  "contract": {
    "name": "...",
    "version": "1.0.0",
    "description": "..."
  },
  "source": {
    "format": "csv",
    "delimiter": ",",
    "has_header": true
  },
  "schema": {
    "fields": {
      "Student Name": {
        "type": "string",
        "nullable": false
      },
      "Class": {
        "type": "string",
        "nullable": false,
        "allowed_values": [
          "Calculus 1",
          "English 101",
          "Python 101",
          "Intro to Databases"
        ]
      },
      "Semester": {
        "type": "string",
        "nullable": false,
        "allowed_values": [
          "Fall",
          "Spring"
        ]
      },
      "Grade": {
        "type": "integer",
        "nullable": false,
        "min_value": 60,
        "max_value": 100
      }
    }
  },
  "quality_rules": {
    "no_duplicate_rows": true,
    "min_row_count": 1
  }
}
```

---

## Part 2 — Validate the Dataset in Python

Write a Python script or Jupyter notebook that validates the dataset against your contract.

Your code must check the following, in order:

1. Load the CSV using `pandas`
2. Load the JSON contract using the `json` library
3. Check that all required columns are present
4. Check that no column contains null or empty values
5. Check that all `Class` values match the allowed list
6. Check that all `Semester` values match the allowed list
7. Check that all `Grade` values are integers within range
8. Check for duplicate rows

For each check, print a clear `PASS` or `FAIL`.

If a check fails, print the rows that caused the failure.

---

## Deliverable

Submit the following file:

```text
student_grades_contract.json
```

> Make sure you use this exact file name.

---

## Tips

- Read the full dataset before writing the contract.
- Your contract is a structured JSON file.
- Test your validation script on the provided dataset.
- There should be **4 failed rows** if your validation is working correctly.

In [ ]:
df = pd.read_csv("student_grades.csv")
print(f"Rows loaded: {len(df)}")
df.head()



#check colu
missing = [c for c in contract["columns"] if c not in df.columns]

if missing:
    print(f"FAIL — Missing columns: {missing}")
else:
    print("PASS — All expected columns present")

## check for nulls
null_counts = df[contract["columns"]].isnull().sum()

if null_counts.sum() == 0:
    print("PASS — No null values found")
else:
    print("FAIL — Null values found:")
    print(null_counts[null_counts > 0])




In [ ]:
## check allowed values
bad_classes = df[~df["Class"].isin(contract["allowed_classes"])]
bad_semesters = df[~df["Semester"].isin(contract["allowed_semesters"])]

if bad_classes.empty:
    print("PASS — All Class values are valid")
else:
    print(f"FAIL — {len(bad_classes)} invalid Class value(s):")
    print(bad_classes[["Student Name", "Class"]])

if bad_semesters.empty:
    print("PASS — All Semester values are valid")
else:
    print(f"FAIL — {len(bad_semesters)} invalid Semester value(s):")
    print(bad_semesters[["Student Name", "Semester"]])

In [ ]:
## cehck grade range
bad_grades = df[(df["Grade"] < contract["grade_min"]) | (df["Grade"] > contract["grade_max"])]

if bad_grades.empty:
    print(f"PASS — All grades are between {contract['grade_min']} and {contract['grade_max']}")
else:
    print(f"FAIL — {len(bad_grades)} out-of-range grade(s):")
    print(bad_grades[["Student Name", "Grade"]])

In [ ]:
checks = {
    "No missing columns":   len(missing) == 0,
    "No null values":        null_counts.sum() == 0,
    "Valid Class values":    bad_classes.empty,
    "Valid Semester values": bad_semesters.empty,
    "Grades in range":       bad_grades.empty,
}

failed_records = {
    "Valid Class values":    bad_classes,
    "Valid Semester values": bad_semesters,
    "Grades in range":       bad_grades,
}

print("\n--- Validation Summary ---")
for check, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {check}")
    if not passed and check in failed_records:
        records = failed_records[check]
        if not records.empty:
            print(records.to_string(index=False))
            print()

overall = all(checks.values())
print(f"\nOverall: {'✓ ALL CHECKS PASSED' if overall else '✗ VALIDATION FAILED'}")